# 📊 Household Income — HIES (National & State)

## Dataset Description
This notebook loads two related datasets from DOSM's **Household Income and Expenditure Survey (HIES/HIS)**: national-level household income trends and state-level breakdowns. These are collected every 2–3 years and represent the most authoritative income distribution data in Malaysia.

## Data Sources
| Dataset | Catalogue Page | Parquet URL |
|---|---|---|
| National household income | https://open.dosm.gov.my/data-catalogue/hh_income | `storage.dosm.gov.my/hies/hh_income.parquet` |
| Household income by state | https://open.dosm.gov.my/data-catalogue/hh_income_state | `storage.dosm.gov.my/hies/hh_income_state.parquet` |
| Income by percentile | https://open.dosm.gov.my/data-catalogue/hies_malaysia_percentile | `storage.dosm.gov.my/hies/hies_malaysia_percentile.parquet` |
| Gini coefficient | https://open.dosm.gov.my/data-catalogue/hh_inequality | `storage.dosm.gov.my/hies/hh_inequality.parquet` |

**License:** CC BY 4.0 — DOSM  
**Coverage:** 1970 – 2022 (HIES years: 1970, 1974, 1976, 1979, 1982, 1984, 1987, 1989, 1992, 1995, 1997, 1999, 2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2022)

## Column Descriptions — National Income
| Column | Type | Description |
|---|---|---|
| `date` | date | Survey year |
| `income_mean` | float | Mean gross monthly household income (RM) |
| `income_median` | float | Median gross monthly household income (RM) |

## Column Descriptions — State Income
| Column | Type | Description |
|---|---|---|
| `date` | date | Survey year |
| `state` | string | Malaysian state name |
| `income_mean` | float | Mean gross monthly household income (RM) |
| `income_median` | float | Median gross monthly household income (RM) |
| `expenditure_mean` | float | Mean monthly household expenditure (RM) |
| `gini` | float | Gini coefficient (0 = perfect equality, 1 = perfect inequality) |
| `poverty` | float | Poverty incidence rate (%) |

## Relevance to EduNilai
HIES income data provides the **counterfactual baseline** for ROI: what does a non-graduate household earn? The state breakdown supports the geographic inequality dimension. The Gini trend shows whether expanding higher education has actually reduced income inequality.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# National household income
df_national = pd.read_parquet('https://storage.dosm.gov.my/hies/hh_income.parquet')
if 'date' in df_national.columns:
    df_national['date'] = pd.to_datetime(df_national['date'])

# State-level income
df_state = pd.read_parquet('https://storage.dosm.gov.my/hies/hh_income_state.parquet')
if 'date' in df_state.columns:
    df_state['date'] = pd.to_datetime(df_state['date'])

# Gini coefficient
df_gini = pd.read_parquet('https://storage.dosm.gov.my/hies/hh_inequality.parquet')
if 'date' in df_gini.columns:
    df_gini['date'] = pd.to_datetime(df_gini['date'])

print("National:", df_national.shape, "| cols:", list(df_national.columns))
print("State:   ", df_state.shape,    "| cols:", list(df_state.columns))
print("Gini:    ", df_gini.shape,     "| cols:", list(df_gini.columns))

In [ ]:
# Filter national to 2009 onwards (HIES years available)
df_nat_2009 = df_national[df_national['date'].dt.year >= 2009].copy()
print("National income (2009 onwards):")
print(df_nat_2009.to_string(index=False))

In [ ]:
# Plot: national median income trend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_nat_2009.set_index('date')[['income_mean', 'income_median']].plot(ax=axes[0], marker='o')
axes[0].set_title('Mean vs Median Household Income (National)')
axes[0].set_ylabel('Monthly Income (RM)')

# Gini trend
df_gini_2009 = df_gini[df_gini['date'].dt.year >= 2009]
if 'gini' in df_gini_2009.columns:
    df_gini_2009.set_index('date')['gini'].plot(ax=axes[1], marker='o', color='darkorange')
    axes[1].set_title('Gini Coefficient (Income Inequality)')
    axes[1].set_ylabel('Gini')
    axes[1].set_ylim(0.3, 0.55)

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('../data/demographic', exist_ok=True)

df_nat_2009.to_csv('../data/demographic/hh_income_national.csv', index=False, encoding='utf-8')
df_state[df_state['date'].dt.year >= 2009].to_csv('../data/demographic/hh_income_state.csv', index=False, encoding='utf-8')
df_gini[df_gini['date'].dt.year >= 2009].to_csv('../data/demographic/income_inequality_gini.csv', index=False, encoding='utf-8')

print("Saved:")
print("  ../data/demographic/hh_income_national.csv")
print("  ../data/demographic/hh_income_state.csv")
print("  ../data/demographic/income_inequality_gini.csv")